# 🔢 손글씨 숫자 인식 AI 모델

## 프로젝트 개요

| 항목 | 내용 |
|------|------|
| **목표** | 0~9 손글씨 숫자 이미지를 입력받아 어떤 숫자인지 자동 분류 |
| **AI 프레임워크** | TensorFlow / Keras |
| **데이터셋** | EMNIST Digits (28만 장) — MNIST의 4배 |
| **모델 구조** | CNN (합성곱 신경망) |
| **예상 정확도** | 99.0% ~ 99.5% |

---

### ⚙️ 사전 설정
이 노트북을 실행하기 전에 **GPU를 활성화**하세요:
1. 상단 메뉴 → **런타임** → **런타임 유형 변경**
2. 하드웨어 가속기 → **T4 GPU** 선택 → **저장**

## 1단계: 환경 확인

먼저 필요한 라이브러리를 불러오고, GPU가 정상적으로 연결되었는지 확인합니다.

| 라이브러리 | 역할 | 비유 |
|-----------|------|------|
| `tensorflow` | AI 모델을 만들고 학습시키는 핵심 도구 | 요리사의 주방 |
| `numpy` | 숫자 데이터를 빠르게 처리 | 재료 손질 도구 |
| `matplotlib` | 그래프와 이미지를 화면에 표시 | 요리 결과물 사진 |

In [ ]:
# 라이브러리 불러오기
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 버전 및 GPU 확인
print(f"TensorFlow 버전: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU 사용 가능: {gpu_devices[0].name}")
else:
    print("⚠️ GPU가 감지되지 않았습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.")

## 2단계: 데이터 준비

**EMNIST Digits** 데이터셋을 다운로드합니다. (총 28만 장)

> **AI 학습은 학생이 공부하는 것과 같습니다:**
> - `x_train` / `y_train` = 연습 문제 + 해답 (공부용, 24만 장)
> - `x_test` / `y_test` = 시험 문제 + 해답 (실력 평가용, 4만 장)
>
> AI는 연습 문제로 공부한 뒤, 한 번도 본 적 없는 시험 문제로 실력을 평가받습니다.

### 데이터셋 비교

| 데이터셋 | 이미지 수 | 특징 |
|----------|-----------|------|
| MNIST | 70,000장 | 가장 유명하지만 데이터가 적음 |
| **EMNIST Digits** | **280,000장** | **MNIST의 4배, 다양한 필체 포함 ← 선택!** |
| SVHN | 600,000장 | 실제 사진 (난이도 높음, 초보자 비추천) |

In [ ]:
import tensorflow_datasets as tfds

# EMNIST Digits 데이터셋 다운로드 (처음 실행 시 약 1~2분 소요)
print("EMNIST Digits 데이터셋 다운로드 중...")
emnist = tfds.load('emnist/digits', as_supervised=True)
train_ds = emnist['train']
test_ds = emnist['test']

# numpy 배열로 변환
print("데이터 변환 중...")
x_train, y_train = [], []
for image, label in train_ds:
    x_train.append(image.numpy())
    y_train.append(label.numpy())
x_train = np.array(x_train)
y_train = np.array(y_train)

x_test, y_test = [], []
for image, label in test_ds:
    x_test.append(image.numpy())
    y_test.append(label.numpy())
x_test = np.array(x_test)
y_test = np.array(y_test)

# ★ EMNIST 회전 보정 (중요!)
# EMNIST 원본 데이터는 90도 회전 + 좌우 반전된 상태로 저장되어 있음
# 사람이 자연스럽게 쓰는 방향으로 보정해야 직접 그린 숫자와 매칭됨
print("EMNIST 회전 보정 중...")
x_train = np.array([np.fliplr(np.rot90(img.squeeze(), k=3))[:,:,np.newaxis] for img in x_train])
x_test = np.array([np.fliplr(np.rot90(img.squeeze(), k=3))[:,:,np.newaxis] for img in x_test])

# 데이터 정보 출력
print(f"\n✅ 다운로드 완료!")
print(f"   학습 데이터: {x_train.shape} → {len(x_train):,}장의 이미지")
print(f"   테스트 데이터: {x_test.shape} → {len(x_test):,}장의 이미지")
print(f"   이미지 크기: {x_train.shape[1]}x{x_train.shape[2]} 픽셀 (흑백)")
print(f"   라벨 종류: {np.unique(y_train)} → 0~9 숫자")
print(f"   ★ 회전 보정 적용됨")

## 3단계: 데이터 전처리 (정규화)

이미지 픽셀 값을 **0~255**에서 **0~1** 범위로 변환합니다.

> **왔 정규화를 하나요?**
>
> | 상태 | 픽셀 범위 | AI 학습 효과 |
> |------|-----------|-------------|
> | 정규화 전 | 0 ~ 255 | 큰 숫자 때문에 학습이 불안정하고 느림 |
> | **정규화 후** | **0.0 ~ 1.0** | **작은 숫자로 학습이 안정적이고 빠름** |
>
> 비유: 시험 점수를 100점 만점에서 1점 만점으로 바꾸는 것과 같습니다.
> 85점 → 0.85로 바꿔도 의미는 같지만, 계산이 더 쉬워집니다.

In [ ]:
# 픽셀 값을 0~1 범위로 정규화
# 원래 픽셀: 0(검정) ~ 255(흰색) → 변환 후: 0.0 ~ 1.0
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"정규화 완료!")
print(f"  픽셀 범위: {x_train.min():.1f} ~ {x_train.max():.1f}")
print(f"  학습 데이터 shape: {x_train.shape}")
print(f"  테스트 데이터 shape: {x_test.shape}")

## 4단계: 데이터 탐색 (시각화)

실제 데이터가 어떻게 생겼는지 눈으로 확인합니다. 숫자 0~9 각각의 샘플 이미지를 표시합니다.

In [ ]:
# Display sample images for digits 0~9
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('EMNIST Digits Sample Images (0~9)', fontsize=16)

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax = axes[digit // 5][digit % 5]
    ax.imshow(x_train[idx].squeeze(), cmap='gray')
    ax.set_title(f'Digit: {digit}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

# 각 숫자별 데이터 개수 확인
print("Training data count per digit:")
for digit in range(10):
    count = np.sum(y_train == digit)
    bar = '█' * (count // 1000)
    print(f"  Digit {digit}: {count:>6,}  {bar}")
print(f"\n  Total: {len(y_train):,}")

## 5단계: CNN 모델 설계

이미지를 인식할 수 있는 AI의 "뇌 구조"를 설계합니다.
**CNN(합성곱 신경망)**은 이미지 인식에 가장 많이 사용되는 AI 구조입니다.

```
입력 이미지 (28x28)
    ↓
[Conv2D 32필터] → 선, 모서리 같은 기본 특징 추출    (돋보기로 세부 관찰)
    ↓
[MaxPooling]    → 이미지 크기 축소 (14x14)          (요약본 만들기)
    ↓
[Conv2D 64필터] → 곡선, 원 같은 중간 특징 추출       (패턴 인식)
    ↓
[MaxPooling]    → 이미지 크기 축소 (5x5)            (핵심만 남기기)
    ↓
[Conv2D 64필터] → 숫자 전체 형태 파악               (종합 분석)
    ↓
[Flatten]       → 2D → 1D 변환                    (한 줄로 나열)
    ↓
[Dense 128]     → 특징들을 종합 판단                (최종 판단)
    ↓
[Dropout 0.5]   → 과적합 방지                      (암기 방지)
    ↓
[Dense 10]      → 0~9 각 숫자일 확률 출력           (정답 선택)
```

> **과적합(Overfitting)이란?**
> 학습 데이터를 너무 외워버려서 새로운 데이터는 못 맞추는 현상.
> 시험 범위만 달달 외운 학생이 응용 문제를 못 푸는 것과 같습니다.
> **Dropout**은 학습 중 뉴런 일부를 랜덤으로 꺼서 이를 방지합니다.

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    # --- 첫 번째 특징 추출 블록 ---
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),

    # --- 두 번째 특징 추출 블록 ---
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # --- 세 번째 특징 추출 블록 ---
    layers.Conv2D(64, (3, 3), activation='relu'),

    # --- 분류기 ---
    layers.Flatten(),                        # 2D → 1D 변환
    layers.Dense(128, activation='relu'),    # 종합 판단
    layers.Dropout(0.5),                     # 과적합 방지
    layers.Dense(10, activation='softmax')   # 0~9 확률 출력
])

# 모델 구조 요약 출력
model.summary()

## 6단계: 모델 컴파일 및 학습

설계한 모델에 학습 방법을 설정하고, 실제로 학습을 시작합니다.

| 용어 | 의미 | 비유 |
|------|------|------|
| **optimizer='adam'** | 학습 속도와 방향을 자동 조절 | 자동 기어 자동차 |
| **loss** | AI 예측이 정답과 얼마나 다른지 측정 | 시험 오답 개수 |
| **epochs=10** | 전체 데이터를 10번 반복 학습 | 교과서를 10번 읽기 |
| **batch_size=128** | 한 번에 128장씩 묶어 학습 | 128문제씩 풀고 채점 |
| **validation_split=0.1** | 학습 데이터의 10%로 중간 테스트 | 모의고사 |

> GPU 사용 시 약 **5~10분** 소요됩니다.

In [ ]:
# 모델 컴파일 (학습 방법 설정)
model.compile(
    optimizer='adam',                            # 최적화 알고리즘
    loss='sparse_categorical_crossentropy',      # 손실 함수
    metrics=['accuracy']                         # 평가 지표: 정확도
)

# 모델 학습 시작!
print("🚀 학습을 시작합니다...\n")
history = model.fit(
    x_train, y_train,       # 학습 데이터 (문제 + 정답)
    epochs=10,               # 전체 데이터를 10번 반복
    batch_size=128,          # 한 번에 128장씩 학습
    validation_split=0.1,    # 10%는 검증용으로 분리
    verbose=1                # 진행 상황 표시
)
print("\n✅ 학습 완료!")

## 7단계: 성능 평가

학습된 모델이 **한 번도 본 적 없는 테스트 데이터**에서 얼마나 잘 작동하는지 확인합니다.

**그래프 읽는 법:**
| 그래프 | 좋은 상태 | 나쁜 상태 (과적합) |
|--------|----------|-------------------|
| 정확도 | 학습/검증 모두 높고 비슷함 | 학습은 높은데 검증이 낮음 |
| 손실 | 학습/검증 모두 낮고 비슷함 | 학습은 낮은데 검증이 높음 |

In [ ]:
# Final evaluation on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"{'='*40}")
print(f"  Test Accuracy: {test_accuracy*100:.2f}%")
print(f"  Test Loss: {test_loss:.4f}")
print(f"{'='*40}")

# Training history graphs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy graph
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, marker='s')
axes[0].set_title('Accuracy per Epoch', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Loss graph
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2, marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, marker='s')
axes[1].set_title('Loss per Epoch', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8단계: 결과 시각화

AI가 맞춘 것과 틀린 것을 눈으로 확인합니다. 틀린 사례를 보면 AI가 어떤 숫자를 헷갈려하는지 알 수 있습니다.

In [ ]:
# Predictions on test data
predictions = model.predict(x_test)

# --- Correct predictions (10 examples) ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Correct Predictions', fontsize=16, color='green')

correct_idx = np.where(np.argmax(predictions, axis=1) == y_test)[0]
for i, ax in enumerate(axes.flat):
    idx = correct_idx[i]
    ax.imshow(x_test[idx].squeeze(), cmap='gray')
    pred = np.argmax(predictions[idx])
    conf = np.max(predictions[idx]) * 100
    ax.set_title(f'Pred: {pred} ({conf:.1f}%)', fontsize=11, color='green')
    ax.axis('off')
plt.tight_layout()
plt.show()

# --- Wrong predictions (10 examples) ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Wrong Predictions', fontsize=16, color='red')

wrong_idx = np.where(np.argmax(predictions, axis=1) != y_test)[0]
for i, ax in enumerate(axes.flat):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        ax.imshow(x_test[idx].squeeze(), cmap='gray')
        pred = np.argmax(predictions[idx])
        true = y_test[idx]
        ax.set_title(f'Pred: {pred} / True: {true}', fontsize=11, color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nTotal test images: {len(y_test):,}")
print(f"  Correct: {len(correct_idx):,}")
print(f"  Wrong:   {len(wrong_idx):,}")

## 9단계: 혼동 행렬 (Confusion Matrix)

어떤 숫자를 어떤 숫자로 잘못 인식하는지 패턴을 파악합니다.

> **읽는 법:**
> - **대각선** 숫자가 클수록 좋음 (정답 개수)
> - **대각선 밖** 숫자는 오답 (예: 행=4, 열=9이면 "4를 9로 잘못 인식")
> - 흔한 혼동 패턴: 4↔9, 3↔8, 1↔7

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Confusion Matrix
y_pred = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_test, y_pred)

# Heatmap visualization
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## 10단계: 모델 저장

학습된 모델을 파일로 저장합니다. Google Drive에 백업하면 Colab 세션이 종료되어도 모델이 보존됩니다.

> **완 저장하나요?**
> Colab은 임시 환경이라 세션이 끝나면 모든 파일이 삭제됩니다.
> Google Drive에 저장하면 다음에 학습 없이 바로 사용할 수 있습니다.

In [ ]:
# Colab 환경에 모델 저장
model.save('digit_recognition_model.keras')
print("✅ 모델 저장 완료: digit_recognition_model.keras")

# Google Drive에 백업 (영구 보관)
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('digit_recognition_model.keras',
            '/content/drive/MyDrive/digit_recognition_model.keras')
print("✅ Google Drive에 백업 완료!")
print("   위치: Google Drive > 내 드라이브 > digit_recognition_model.keras")

## 11단계: 직접 그린 숫자로 테스트

직접 숫자 이미지를 업로드하여 AI의 예측을 확인해 봅니다.

**테스트 방법:**
1. 그림판에서 **검은 배경에 흰색 글씨**로 숫자를 그림
2. 이미지를 PNG/JPG로 저장
3. 아래 셀을 실행하면 파일 업로드 버튼이 나타남
4. 이미지를 업로드하면 AI가 예측 결과를 표시

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
import io

def preprocess_drawn_image(image_bytes):
    """
    그림판에서 그린 이미지를 EMNIST 학습 데이터와 동일한 형태로 전처리

    처리 과정:
    1. 흑백 변환
    2. 색상 반전 (흰 배경+검은 글씨 → 검은 배경+흰 글씨)
    3. 숫자 영역만 잘라내기 (crop)
    4. 정사각형으로 패딩
    5. 20x20으로 리사이즈 후 28x28 중앙 배치 (EMNIST 방식)
    6. 정규화 (0~1)
    """
    # 이미지 로드 및 흑백 변환
    img = Image.open(io.BytesIO(image_bytes)).convert('L')

    # 색상 반전 (그림판: 흰 배경+검은 글씨 → EMNIST: 검은 배경+흰 글씨)
    img = ImageOps.invert(img)

    # numpy 배열로 변환
    img_array = np.array(img)

    # 숫자가 있는 영역만 잘라내기 (bounding box)
    coords = np.argwhere(img_array > 30)  # 임계값 이상인 픽셀 좌표
    if len(coords) == 0:
        # 빈 이미지인 경우
        return np.zeros((1, 28, 28, 1), dtype='float32')

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    cropped = img_array[y_min:y_max+1, x_min:x_max+1]

    # 정사각형으로 패딩 (긴 쪽에 맞춤)
    h, w = cropped.shape
    max_side = max(h, w)
    padded = np.zeros((max_side, max_side), dtype=np.uint8)
    y_offset = (max_side - h) // 2
    x_offset = (max_side - w) // 2
    padded[y_offset:y_offset+h, x_offset:x_offset+w] = cropped

    # 20x20으로 리사이즈 (EMNIST는 20x20 숫자를 28x28 중앙에 배치)
    img_resized = Image.fromarray(padded).resize((20, 20), Image.LANCZOS)
    img_20 = np.array(img_resized)

    # 28x28 캔버스 중앙에 배치 (4px 여백)
    canvas = np.zeros((28, 28), dtype=np.uint8)
    canvas[4:24, 4:24] = img_20

    # 정규화
    result = canvas.astype('float32') / 255.0
    return result.reshape(1, 28, 28, 1)

# --- 파일 업로드 및 테스트 ---
print("Upload a digit image (PNG/JPG)")
print("You can draw on ANY background - preprocessing handles it automatically.\n")
uploaded = files.upload()

for filename in uploaded.keys():
    # 전처리
    img_array = preprocess_drawn_image(uploaded[filename])

    # 전처리 결과 확인 (Before/After)
    original = Image.open(io.BytesIO(uploaded[filename])).convert('L')
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(np.array(original), cmap='gray')
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(img_array.squeeze(), cmap='gray')
    axes[1].set_title('Preprocessed (28x28)', fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # 예측
    prediction = model.predict(img_array, verbose=0)
    predicted_digit = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    # 결과 표시
    plt.figure(figsize=(4, 4))
    plt.imshow(img_array.squeeze(), cmap='gray')
    plt.title(f'Prediction: {predicted_digit} (Confidence: {confidence:.1f}%)', fontsize=14)
    plt.axis('off')
    plt.show()

    # 각 숫자별 확률
    print(f"\nProbability per digit:")
    for i in range(10):
        bar = '█' * int(prediction[0][i] * 50)
        marker = ' << Predicted' if i == predicted_digit else ''
        print(f"  {i}: {prediction[0][i]*100:5.1f}% {bar}{marker}")
    print()

---

## 📖 용어 사전

| 용어 | 영어 | 설명 |
|------|------|------|
| 합성곱 신경망 | CNN | 이미지 인식에 특화된 AI 구조 |
| 에폭 | Epoch | 전체 데이터를 1회 학습하는 단위 |
| 배치 | Batch | 한 번에 학습하는 데이터 묶음 |
| 과적합 | Overfitting | 학습 데이터만 잘 맞추고 새 데이터는 못 맞추는 현상 |
| 정규화 | Normalization | 데이터 값 범위를 일정하게 조정 |
| 손실 함수 | Loss Function | AI 예측의 오차를 측정하는 함수 |
| 활성화 함수 | Activation Function | 뉴런의 출력을 결정하는 함수 (ReLU, Softmax 등) |
| 드롭아웃 | Dropout | 과적합 방지를 위해 뉴런을 랜덤으로 비활성화 |
| 소프트맥스 | Softmax | 출력을 확률(합=100%)로 변환하는 함수 |
| 혼동 행렬 | Confusion Matrix | 예측 결과의 정답/오답 분포를 보여주는 표 |

---

## 🚀 다음 단계 (심화 과제)

| 단계 | 과제 | 난이도 |
|------|------|--------|
| 심화 1 | Data Augmentation (이미지 회전/이동으로 데이터 늘리기) | ★★☆☆ |
| 심화 2 | SVHN 데이터셋으로 실제 사진 속 숫자 인식 | ★★★☆ |
| 심화 3 | 웹캠으로 실시간 숫자 인식 | ★★★★ |
| 심화 4 | TensorFlow.js로 웹 앱 배포 | ★★★★ |
| 심화 5 | 다중 숫자 인식 (번호판 등) | ★★★★★ |